# 05. Brecha Léxica CO vs Core vs PY

**Objetivo:** justificar científicamente la transición `Concept_CO -> Concept_Core -> Concept_PY`.
**Entradas (inputs):** `data/splits/dataset_base.csv`, `data/splits/*_indices.csv`, `data/processed/gemini_extraction.json` (opcional para auditoría).
**Salidas (outputs):** tablas comparativas por perfil (`co/core/py`) y evidencia para capítulo metodológico.
**Notebook anterior:** `notebooks/pipeline/04b_linea_base_tfidf.ipynb` y `notebooks/pipeline/04c_linea_base_transformers.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/06_ingenieria_features_hibridas.ipynb`.

**Puntos metodológicos obligatorios en este notebook:**
1. insuficiencia de `Concept_CO` para EHR paraguayo,
2. depuración de `Concept_Core`,
3. construcción de `Concept_PY`,
4. apoyo de LLM para verificar colombianismos y variantes paraguayas (sin reemplazar reglas congeladas).


## Marco metodológico de la brecha léxica

1. **Por qué `Concept_CO` es insuficiente en Paraguay**
   - Fue diseñado con variantes colombianas y no cubre de forma robusta jopará, abreviaturas IPS ni giros locales.
2. **Cómo se depuró `Concept_Core`**
   - Se eliminaron disparadores no clínicos, tokens ambiguos y reglas con alto riesgo de falso positivo.
3. **Cómo se expandió `Concept_PY`**
   - Se incorporaron variantes paraguayas, abreviaturas institucionales y errores ortográficos frecuentes con anclaje clínico.
4. **Cómo se detectaron variantes regionales y abreviaturas**
   - Se combinaron auditoría de corpus, revisión experta y comparación de perfiles (`co/core/py`) en el mismo subset.

### Uso del LLM en la auditoría léxica
Durante la fase de auditoría léxica, el LLM se utilizó como apoyo metodológico para verificar:
- expresiones coloquiales,
- colombianismos presentes en `Concept_CO`,
- variantes paraguayas detectadas en el corpus,
- equivalencias semánticas entre variantes.

El LLM **no reemplaza** reglas clínicas deterministas: se usa como soporte de normalización y contraste semántico.


In [ ]:
import sys, time, json, re
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Utilidades compartidas del proyecto
current_dir = Path.cwd().resolve()
if current_dir.name in {'pipeline', 'analysis', 'appendix'}:
    REPO_ROOT = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from notebooks.utils_shared import setup_paths, load_splits, guess_text_col, make_run_id, get_output_dir, export_fp_fn_candidates
except ImportError:
    from utils_shared import setup_paths, load_splits, guess_text_col, make_run_id, get_output_dir, export_fp_fn_candidates

paths = setup_paths()
BASE_PATH = paths['BASE_PATH']
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
FORK_PATH = paths['FORK_PATH']

print('BASE_PATH:', BASE_PATH)
print('FORK_PATH:', FORK_PATH)

# Importar build_pipeline desde el fork clínico
if str(FORK_PATH) not in sys.path:
    sys.path.insert(0, str(FORK_PATH))
from cli import build_pipeline, load_yaml


In [ ]:
# =====================
# Parámetros de ejecución
# =====================
PROFILES = ['co','core','py']  # baseline, core, core+PY
# PROFILES=['core','py']  # perfiles evaluados en la auditoría
SPLIT = 'dev'  # 'train' | 'dev' | 'test' | 'all'
SAMPLE_MAX = None  # ej. 2000 para pruebas rápidas
BATCH_SIZE = 128
N_PROCESS = 1  # ajusta según tu CPU; 1 si hay problemas

RUN_ID = make_run_id(prefix='compare')
print('RUN_ID:', RUN_ID)

# Archivo Gemini (si existe)
GEMINI_JSON = DATA_PATH / 'processed' / 'gemini_extraction.json'
if not GEMINI_JSON.exists():
    # Alternativa de respaldo para nombres históricos
    for alt in ['gemini_extractions.json','gemini_output.json']:
        p = DATA_PATH / 'processed' / alt
        if p.exists():
            GEMINI_JSON = p
            break
print('Gemini JSON:', GEMINI_JSON)


In [ ]:
# =====================
# Cargar conjunto de datos base
# =====================
df_base, train_idx, dev_idx, test_idx = load_splits(SPLITS_PATH)
text_col = guess_text_col(df_base)
print('text_col:', text_col)

if SPLIT == 'train':
    ids = set(train_idx)
elif SPLIT == 'dev':
    ids = set(dev_idx)
elif SPLIT == 'test':
    ids = set(test_idx)
else:
    ids = set(df_base['row_id'].tolist())

df = df_base[df_base['row_id'].isin(ids)][['row_id', text_col]].copy()
df[text_col] = df[text_col].fillna('').astype(str)

if SAMPLE_MAX is not None and len(df) > SAMPLE_MAX:
    df = df.sample(SAMPLE_MAX, random_state=42).sort_values('row_id').reset_index(drop=True)

print(f'Dataset para comparación: {len(df)} filas (split={SPLIT})')


In [ ]:
# =====================
# Función: extraer matriz de reglas por perfil con `nlp.pipe`
# =====================
from utils_shared import keep_entity  # <-- ahora viene del shared

def extract_rule_matrix(nlp, df, text_col, batch_size=128, n_process=1,
                        window_tokens=12, add_patient_neg_features=True):
    rows = df[['row_id', text_col]].copy()
    texts = rows[text_col].fillna('').astype(str).tolist()
    row_ids = rows['row_id'].tolist()

    rule_rows = []
    seen_cats = set()

    for rid, doc in tqdm(zip(row_ids, nlp.pipe(texts, batch_size=batch_size, n_process=n_process)),
                         total=len(row_ids)):
        d = {'row_id': int(rid)}

        for ent in getattr(doc, "ents", []):
            keep, is_pat_neg = keep_entity(ent, doc, window_tokens=window_tokens)
            if not keep:
                continue

            # Regla afirmada o negación del paciente: ambas se conservan como señal clínica (keep=True)
            col = f"rule_{ent.label_}"
            d[col] = 1
            seen_cats.add(ent.label_)   # BUG FIX: ahora sí se llena

            # Característica adicional: `niega_*` para negación subjetiva del paciente
            if add_patient_neg_features and is_pat_neg:
                d[f"niega_{ent.label_}"] = 1

        rule_rows.append(d)

    df_rules = pd.DataFrame(rule_rows).fillna(0).astype({k: int for k in rule_rows[0].keys() if k != 'row_id'})

    # Asegurar columnas 0/1 para todas las categorías observadas (`rule_*`)
    for cat in sorted(seen_cats):
        col = f"rule_{cat}"
        if col not in df_rules.columns:
            df_rules[col] = 0
        if add_patient_neg_features:
            ncol = f"niega_{cat}"
            if ncol not in df_rules.columns:
                df_rules[ncol] = 0

    # Asegurar tipo entero
    for c in df_rules.columns:
        if c != "row_id":
            df_rules[c] = df_rules[c].astype(int)

    return df_rules


In [ ]:
def export_support_and_samples(df_text, df_rules, out_dir, text_col="texto", sample_n=20, random_state=42):
    """
    Exporta:
      - phenotype_support.csv: soporte (#matches) por fenotipo
      - samples_by_phenotype/<Phenotype>.csv: N ejemplos (row_id + texto) por fenotipo

    df_text: dataframe original con ['row_id', text_col]
    df_rules: matriz de reglas con ['row_id', 'rule_*'...]
    """
    out_dir = Path(out_dir)
    samples_dir = out_dir / "samples_by_phenotype"
    samples_dir.mkdir(parents=True, exist_ok=True)

    # Unir texto clínico con activaciones `rule_*`
    use_cols = ["row_id", text_col]
    df_m = df_rules.merge(df_text[use_cols], on="row_id", how="left")

    rule_cols = [c for c in df_rules.columns if c.startswith("rule_")]

    # Número de reglas activadas por nota (útil para auditoría)
    df_m["n_rules"] = df_m[rule_cols].sum(axis=1)

    # Soporte por fenotipo
    support_rows = []
    for col in rule_cols:
        phen = col.replace("rule_", "")
        support = int(df_rules[col].sum())
        support_rows.append({"phenotype": phen, "rule_col": col, "support": support})

    df_support = pd.DataFrame(support_rows).sort_values("support", ascending=False)
    df_support.to_csv(out_dir / "phenotype_support.csv", index=False, encoding="utf-8-sig")

    # Muestras por fenotipo (solo con soporte > 0)
    rng = random_state
    for _, r in df_support[df_support["support"] > 0].iterrows():
        col = r["rule_col"]
        phen = r["phenotype"]

        hits = df_m[df_m[col] == 1][["row_id", "n_rules", text_col]].dropna()

        # Muestreo estable (semilla fija)
        if len(hits) > sample_n:
            hits = hits.sample(sample_n, random_state=rng)

        hits = hits.copy()
        hits.insert(0, "phenotype", phen)
        hits.to_csv(samples_dir / f"{phen}.csv", index=False, encoding="utf-8-sig")

    print(f"phenotype_support.csv + samples_by_phenotype/* → {out_dir}")

In [ ]:
# =====================
# Ejecutar extracción por perfiles
# =====================
config_path = FORK_PATH / 'configs' / 'fenotipos.yml'
# Configuración de fenotipos (categorías clínicas disponibles)
fenos_cfg = load_yaml(config_path)

# Configuración por perfil (carpetas y capas a cargar)
def load_profile_cfg(profile: str):
    p = FORK_PATH / "configs" / f"{profile}_config.yml"
    if not p.exists():
        raise FileNotFoundError(f"No existe config de perfil: {p}")
    return load_yaml(p)

summary = []
rule_matrices = {}

for profile in PROFILES:
    print('\n' + '='*80)
    print('PROFILE:', profile)
    out_dir = get_output_dir(BASE_PATH, profile, RUN_ID)
    print('OUT_DIR:', out_dir)

    t0 = time.time()
    profile_cfg = load_profile_cfg(profile)
    nlp = build_pipeline(profile, fenos_cfg, profile_cfg)
    
    # --- Depuración opcional
    print("PROFILE", profile)
    print("patterns_root:", profile_cfg["patterns_root"])
    print("concept_layers:", profile_cfg["concept_layers"])

    from pathlib import Path
    root = FORK_PATH / profile_cfg["patterns_root"]
    print("resolved patterns root:", root, "| exists:", root.exists())
    for layer in profile_cfg["concept_layers"]:
        p = root / layer
        print(" layer:", layer, "->", p, "| exists:", p.exists())

    tm = nlp.get_pipe("medspacy_target_matcher")

    cands = [
        ("target_rules", getattr(tm, "target_rules", None)),
        ("_target_rules", getattr(tm, "_target_rules", None)),
        ("rules", getattr(tm, "rules", None)),
    ]
    for name, obj in cands:
        if obj is None:
            continue
        try:
            print(profile, name, "len =", len(obj))
        except TypeError:
            print(profile, name, "type =", type(obj))

    # Imprimir categorías cuando estén disponibles
    cats = getattr(tm, "rule_categories", None) or getattr(tm, "categories", None)
    if cats is not None:
        try:
            print(profile, "categories len =", len(cats))
        except TypeError:
            print(profile, "categories type =", type(cats))
    # ---

    t1 = time.time()

    df_rules = extract_rule_matrix(nlp, df, text_col, batch_size=BATCH_SIZE, n_process=N_PROCESS)
    # --- Depuración
    cols = df_rules.columns.tolist()

    print("n_cols:", len(cols))
    print("n_rule_cols:", sum(c.startswith("rule_") for c in cols))
    print("n_niega_cols:", sum(c.startswith("niega_") for c in cols))

    # Control rápido: `niega_*` debería ser >0 si `add_patient_neg_features` está activo
    # Las columnas `rule_*` deben mantenerse consistentes entre perfiles (al menos en la unión)

    # Verificar diferencias reales entre `py` y `core` en columnas
    print("any niega? ", any(c.startswith("niega_") for c in cols))
    #---
    t2 = time.time()

    # Métricas descriptivas rápidas
    rule_cols = [c for c in df_rules.columns if c.startswith('rule_')]
    df_rules['n_rules'] = df_rules[rule_cols].sum(axis=1)
    pct_any = (df_rules['n_rules']>0).mean() * 100
    avg_rules = df_rules['n_rules'].mean()

    # Guardar matriz de reglas por perfil
    rules_path = out_dir / 'rule_features.parquet'
    df_rules.to_parquet(rules_path, index=False)
    print(f'rule_features → {rules_path.name} | any_rule={pct_any:.1f}% | avg_rules={avg_rules:.2f}')

    # Exportar candidatos FP/FN con apoyo de Gemini (si existe)
    export_fp_fn_candidates(df_text=df, df_rules=df_rules.drop(columns=['n_rules']), gemini_json_path=GEMINI_JSON, out_dir=out_dir, text_col=text_col)

    # Guardar también en CSV (más portable que Parquet)
    df_rules.to_csv(out_dir / "rule_features.csv", index=False, encoding="utf-8-sig")

    # Exportar soporte y muestras (20 por fenotipo) para auditoría
    export_support_and_samples(
        df_text=df,
        df_rules=df_rules.drop(columns=["n_rules"]) if "n_rules" in df_rules.columns else df_rules,
        out_dir=out_dir,
        text_col=text_col,
        sample_n=20,
        random_state=42
    )

    summary.append({
        'profile': profile,
        'n_rows': len(df),
        'load_pipeline_s': round(t1-t0, 2),
        'extract_rules_s': round(t2-t1, 2),
        'pct_any_rule': round(pct_any, 2),
        'avg_rules_per_row': round(avg_rules, 3),
        'n_rule_columns': len(rule_cols)
    })
    rule_matrices[profile] = df_rules

# Resumen de ejecución
df_summary = pd.DataFrame(summary)
display(df_summary)
summary_path = (BASE_PATH / 'data' / 'outputs' / f'{RUN_ID}_summary.csv')
summary_path.parent.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(summary_path, index=False)
print('Summary guardado en:', summary_path)


In [ ]:
# =====================
# Comparación de solapamiento (CO vs CORE vs PY)
# =====================
def any_rule_mask(df_rules):
    rule_cols = [c for c in df_rules.columns if c.startswith('rule_')]
    return (df_rules[rule_cols].sum(axis=1) > 0)

masks = {p: any_rule_mask(rule_matrices[p]) for p in PROFILES}

# Alineación por `row_id`
rid = rule_matrices[PROFILES[0]]['row_id']
aligned = pd.DataFrame({'row_id': rid})
for p in PROFILES:
    aligned[p] = masks[p].astype(int).values

aligned['pattern'] = aligned[PROFILES].astype(str).agg(''.join, axis=1)
counts = aligned['pattern'].value_counts().reset_index()
counts.columns = ['pattern', 'count']
display(counts)

counts_path = (BASE_PATH / 'data' / 'outputs' / f'{RUN_ID}_overlap_counts.csv')
counts.to_csv(counts_path, index=False)
print('overlap counts guardado en:', counts_path)
